# Bringing Sketches to Life with Generative AI

**Author:** Nosa Peter Inwe
**Course:** Computer Vision and Machine Learning II, Université Paris-Est Créteil (UPEC)
**Professor:** Arezoo Ghodsifard

In this notebook I turn three of my own hand drawings (a car, a house and a sword) into finished images with Stable Diffusion 1.5 and a Scribble ControlNet. Every image comes from a real run of the pretrained models; nothing here is faked or pre-made.

I run it on **Google Colab** with a free T4 GPU (`Runtime → Change runtime type → T4 GPU`), top to bottom:

1. Check the GPU.
2. Write my project code into the Colab workspace (it is built into this notebook).
3. Install the diffusion libraries.
4. Upload my three sketches.
5. Dry run: turn each sketch into a white-on-black control map and check it by eye.
6. Generate 15 images: per sketch, one prompt at ControlNet scales 0.00 / 0.65 / 1.10, plus two alternative prompts at 0.65.
7. Look at the comparison sheets and download everything as a zip.

In [ ]:
# 1. GPU check
import subprocess, sys, os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(cmd, cwd=None):
    """I stream the command's output into the notebook, since Jupyter hides child-process output."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")

print("Running on Colab" if IN_COLAB else "Running locally")
try:
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
except Exception as exc:
    print("No NVIDIA GPU found:", exc)
    print("On Colab: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")

In [ ]:
# 2. I write my project code into the workspace (it lives inside this notebook, so there is nothing to upload)
import json
PROJECT_FILES = {
 "src/__init__.py": "\"\"\"My sketch-to-image pipeline (Nosa Peter Inwe, Computer Vision and Machine Learning II, UPEC).\"\"\"\n",
 "src/sketch_pipeline.py": "\"\"\"Sketch preprocessing, a rough structure check and comparison sheets.\n\nBringing Sketches to Life with Generative AI\nAuthor: Nosa Peter Inwe\nCourse: Computer Vision and Machine Learning II, Université Paris-Est Créteil (UPEC)\nProfessor: Arezoo Ghodsifard\n\nThe Scribble ControlNet I use expects white strokes on a black background, so I\nconvert my pen-on-paper photos into that format here. I only transform drawings I\nsupply; nothing in this file makes up a sketch.\n\"\"\"\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom PIL import Image, ImageOps, ImageDraw, ImageFilter, ImageFont\nimport numpy as np\n\n\ndef prepare_sketch(path: str | Path, size: int = 512, threshold: int = 205) -> Image.Image:\n    \"\"\"Turn one photographed sketch into a square white-on-black control map.\n\n    I pad the photo onto a white square instead of stretching it, so the car or\n    sword keeps its proportions. Any pixel darker than `threshold` counts as ink.\n    \"\"\"\n    if size < 64 or size % 8:\n        # Stable Diffusion's VAE downsamples by 8, so the side must divide by 8.\n        raise ValueError(\"size must be >=64 and divisible by 8\")\n    if not 1 <= threshold <= 254:\n        raise ValueError(\"threshold must be in [1, 254]\")\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(f\"Sketch not found: {path}\")\n    with Image.open(path) as raw:\n        # Phone photos often carry a rotation flag; I apply it before anything else.\n        image = ImageOps.exif_transpose(raw).convert(\"RGB\")\n    image.thumbnail((size, size), Image.Resampling.LANCZOS)\n    canvas = Image.new(\"RGB\", (size, size), \"white\")\n    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))\n    grey = np.asarray(ImageOps.grayscale(canvas), dtype=np.uint8)\n    strokes = (grey < threshold).astype(np.uint8) * 255   # dark ink -> white stroke\n    return Image.fromarray(strokes, \"L\").convert(\"RGB\")\n\n\ndef ink_fraction(control: Image.Image) -> float:\n    \"\"\"Share of white pixels in a control map.\n\n    I use it as a sanity check on the photo, not as a quality score: a value near 0\n    means my lines vanished, and a high value usually means a shadow became \"ink\".\n    \"\"\"\n    return float(np.mean(np.asarray(control.convert(\"L\")) > 127))\n\n\ndef sketch_edge_recall(control: Image.Image, generated: Image.Image,\n                       tolerance_pixels: int = 5, edge_threshold: int = 65) -> float:\n    \"\"\"Fraction of my sketch strokes that land within a few pixels of an edge in the output.\n\n    It is a rough helper for comparing strengths, nothing more. A busy texture\n    (brick, foliage) creates edges everywhere and can push the number up without\n    the shape being right, so my own visual ratings carry the real judgement.\n    \"\"\"\n    if not 1 <= tolerance_pixels <= 20:\n        raise ValueError(\"tolerance_pixels must be 1..20\")\n    if not 1 <= edge_threshold <= 254:\n        raise ValueError(\"edge_threshold must be 1..254\")\n    size = control.size\n    target = ImageOps.grayscale(generated.convert(\"RGB\").resize(size, Image.Resampling.LANCZOS))\n    found = target.filter(ImageFilter.FIND_EDGES)\n    edges = Image.fromarray((np.asarray(found) >= edge_threshold).astype(np.uint8) * 255, \"L\")\n    # A max filter grows each edge pixel, which gives me the \"near an edge\" tolerance.\n    nearby = edges.filter(ImageFilter.MaxFilter(2 * tolerance_pixels + 1))\n    strokes = np.asarray(control.convert(\"L\")) > 127\n    if strokes.sum() == 0:\n        return 0.0\n    return float(np.mean((np.asarray(nearby) > 127)[strokes]))\n\n\ndef make_comparison(sketch_path: str | Path, control: Image.Image,\n                    images: list[tuple[str, Image.Image]], out: str | Path) -> None:\n    \"\"\"Save my original sketch, its control map and the generated images in one labelled row.\"\"\"\n    with Image.open(sketch_path) as raw:\n        original = ImageOps.exif_transpose(raw).convert(\"RGB\")\n    cells = [original, control.convert(\"RGB\")] + [im.convert(\"RGB\") for _, im in images]\n    labels = [\"My hand sketch\", \"Scribble control map\"] + [label for label, _ in images]\n    tile, top = 360, 52\n    sheet = Image.new(\"RGB\", (tile * len(cells), tile + top), \"white\")\n    draw = ImageDraw.Draw(sheet)\n    try:\n        font = ImageFont.load_default(size=22)   # Pillow >= 10.1; large enough to read in my report\n    except TypeError:\n        font = ImageFont.load_default()\n    for ix, (im, label) in enumerate(zip(cells, labels)):\n        im = ImageOps.contain(im, (tile - 20, tile - 20), Image.Resampling.LANCZOS)\n        sheet.paste(im, (ix * tile + (tile - im.width) // 2, top + (tile - im.height) // 2))\n        draw.text((ix * tile + 10, 14), label, fill=\"black\", font=font)\n    out = Path(out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(out)\n",
 "src/run_experiments.py": "\"\"\"My experiment runner: sketch -> Scribble ControlNet + Stable Diffusion 1.5 -> images.\n\nBringing Sketches to Life with Generative AI\nAuthor: Nosa Peter Inwe\nCourse: Computer Vision and Machine Learning II, Université Paris-Est Créteil (UPEC)\nProfessor: Arezoo Ghodsifard\n\nI run two experiments per sketch, always with the same seed:\n  1. Control-strength ablation: one prompt at ControlNet scales 0.00, 0.65 and 1.10.\n     At 0.00 the sketch has no influence, so that image is my prompt-only baseline.\n  2. Prompt variation: two extra prompts at the shared scale 0.65, so I can see\n     whether the structure survives a change of style, material or setting.\n\nUsage (from the project root):\n  python -m src.run_experiments --dry-run    # checks my sketches, saves control maps, no download\n  python -m src.run_experiments --generate   # real inference on a CUDA GPU\n\"\"\"\nfrom __future__ import annotations\nimport argparse, csv, hashlib, json, platform\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom importlib.metadata import version, PackageNotFoundError\nfrom src.sketch_pipeline import prepare_sketch, ink_fraction, sketch_edge_recall, make_comparison\n\n\ndef load_settings(path: Path) -> dict:\n    \"\"\"Read experiments.json and stop early on anything that would spoil the comparison.\"\"\"\n    s = json.loads(Path(path).read_text(encoding=\"utf-8\"))\n    required = (\"model_base\", \"controlnet_model\", \"seed\", \"size\", \"num_inference_steps\",\n                \"guidance_scale\", \"experiments\")\n    for key in required:\n        if key not in s:\n            raise ValueError(f\"Missing configuration: {key}\")\n    if not isinstance(s[\"experiments\"], list) or len(s[\"experiments\"]) < 3:\n        raise ValueError(\"I need at least THREE distinct sketch entries\")\n    if int(s[\"size\"]) < 64 or int(s[\"size\"]) % 8:\n        raise ValueError(\"size must be >=64 and divisible by 8\")\n    if int(s[\"num_inference_steps\"]) < 1:\n        raise ValueError(\"num_inference_steps must be positive\")\n    if float(s[\"guidance_scale\"]) < 0:\n        raise ValueError(\"guidance_scale must not be negative\")\n    if not 0 <= float(s.get(\"prompt_variant_strength\", 0.65)) <= 2:\n        raise ValueError(\"prompt_variant_strength must be 0..2\")\n    if len({e.get(\"sketch\") for e in s[\"experiments\"]}) != len(s[\"experiments\"]):\n        raise ValueError(\"Use a different input file for each sketch entry\")\n    for e in s[\"experiments\"]:\n        if not e.get(\"sketch\") or not e.get(\"prompt\") or not e.get(\"control_strengths\"):\n            raise ValueError(\"Each experiment needs sketch, prompt and control_strengths\")\n        if Path(e[\"sketch\"]).name != e[\"sketch\"] or not e[\"sketch\"].lower().endswith((\".png\", \".jpg\", \".jpeg\")):\n            raise ValueError(\"Use a PNG/JPG filename, not a path, in sketch\")\n        strengths = [float(v) for v in e[\"control_strengths\"]]\n        if not all(0 <= v <= 2 for v in strengths):\n            raise ValueError(\"Control strength must be 0..2\")\n        if len(set(strengths)) != len(strengths):\n            raise ValueError(\"Repeated control strength\")\n        variants = e.get(\"prompt_variants\", [])\n        names = [v.get(\"name\") for v in variants]\n        if any(not n or not str(n).isidentifier() for n in names) or len(set(names)) != len(names) or \"main\" in names:\n            raise ValueError(\"Each prompt variant needs a unique simple name (letters, digits, _), not 'main'\")\n        if any(not v.get(\"prompt\") for v in variants):\n            raise ValueError(\"Each prompt variant needs a prompt\")\n    return s\n\n\ndef sha256(path: Path) -> str:\n    \"\"\"Fingerprint of my input file, so anyone can confirm which sketch produced which image.\"\"\"\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef prepare_inputs(s: dict, root: Path, out: Path) -> dict:\n    \"\"\"Build and save a control map for every sketch; refuse to continue if one is missing.\"\"\"\n    out.mkdir(parents=True, exist_ok=True)\n    controls, missing = {}, []\n    for e in s[\"experiments\"]:\n        p = root / \"data\" / \"sketches\" / e[\"sketch\"]\n        if not p.is_file():\n            missing.append(e[\"sketch\"])\n            continue\n        im = prepare_sketch(p, int(s[\"size\"]), int(s.get(\"threshold\", 205)))\n        frac = ink_fraction(im)\n        if not .002 <= frac <= .40:\n            # Almost no ink, or a shadow flooding the page: fix the photo or threshold first.\n            raise ValueError(f\"{e['sketch']}: stroke fraction {frac:.4f} outside 0.002..0.40; \"\n                             \"check the photograph or the threshold\")\n        control_path = out / (Path(e[\"sketch\"]).stem + \"_control.png\")\n        im.save(control_path)\n        controls[e[\"sketch\"]] = (p, im, frac)\n        print(f\"Ready: {p.name}; ink fraction={frac:.3f}; control={control_path.name}\")\n    if missing:\n        raise FileNotFoundError(\"Missing hand sketches: \" + \", \".join(missing))\n    return controls\n\n\ndef load_pipeline(s: dict, allow_cpu: bool = False):\n    \"\"\"Load the pretrained models from Hugging Face. I do not train anything from scratch.\"\"\"\n    import torch\n    from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler\n    has_cuda = torch.cuda.is_available()\n    if not has_cuda and not allow_cpu:\n        raise RuntimeError(\"No CUDA GPU found. Use a GPU (e.g. a Colab T4); --allow-cpu works but is very slow\")\n    dtype = torch.float16 if has_cuda else torch.float32\n    controlnet = ControlNetModel.from_pretrained(s[\"controlnet_model\"], torch_dtype=dtype, use_safetensors=True)\n    pipe = StableDiffusionControlNetPipeline.from_pretrained(\n        s[\"model_base\"], controlnet=controlnet, torch_dtype=dtype, use_safetensors=True)\n    # UniPC gets a clean image in ~25 steps, which keeps the whole run short.\n    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)\n    if has_cuda:\n        pipe.enable_model_cpu_offload()   # keeps only the active sub-model on the GPU\n    else:\n        pipe.to(\"cpu\")\n    return pipe, (\"cuda\" if has_cuda else \"cpu\")\n\n\ndef packages() -> dict:\n    \"\"\"Exact library versions for my reproducibility record.\"\"\"\n    ans = {\"python\": platform.python_version()}\n    for p in (\"torch\", \"diffusers\", \"transformers\", \"accelerate\", \"Pillow\", \"numpy\"):\n        try:\n            ans[p] = version(p)\n        except PackageNotFoundError:\n            ans[p] = \"not installed\"\n    return ans\n\n\ndef export_rows(rows: list[dict], out: Path) -> None:\n    \"\"\"Write the manifest, a summary CSV and a blank sheet for my own ratings.\"\"\"\n    (out / \"run_manifest.json\").write_text(json.dumps(rows, indent=2), encoding=\"utf-8\")\n    keys = [\"sketch\", \"experiment\", \"prompt_variant\", \"control_strength\", \"edge_recall_proxy\", \"result\",\n            \"prompt\", \"seed\", \"steps\", \"guidance_scale\", \"model_base\", \"controlnet_model\"]\n    with (out / \"run_summary.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        w = csv.DictWriter(f, fieldnames=keys, extrasaction=\"ignore\")\n        w.writeheader()\n        w.writerows(rows)\n    with (out / \"human_review_template.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        keys2 = [\"sketch\", \"experiment\", \"prompt_variant\", \"control_strength\", \"structure_1_to_5\",\n                 \"image_quality_1_to_5\", \"prompt_fit_1_to_5\", \"specific_strength\", \"specific_failure\"]\n        w = csv.DictWriter(f, fieldnames=keys2, extrasaction=\"ignore\")\n        w.writeheader()\n        for row in rows:\n            w.writerow(row)\n\n\ndef run_generation(s: dict, root: Path, out: Path, controls: dict, pipe, device: str, generator_factory) -> list[dict]:\n    \"\"\"Generate every image for both experiments and save comparison sheets as I go.\"\"\"\n    rows = []\n    seed = int(s[\"seed\"])\n    variant_strength = float(s.get(\"prompt_variant_strength\", 0.65))\n\n    def generate(e, control, prompt, strength):\n        # A fresh generator with the same seed gives every run the same starting noise,\n        # so the only differences come from the setting I am testing.\n        return pipe(prompt=prompt, negative_prompt=s.get(\"negative_prompt\"), image=control,\n                    width=int(s[\"size\"]), height=int(s[\"size\"]),\n                    num_inference_steps=int(s[\"num_inference_steps\"]),\n                    guidance_scale=float(s[\"guidance_scale\"]),\n                    controlnet_conditioning_scale=strength,\n                    generator=generator_factory(seed)).images[0]\n\n    def record(e, sketch, control, frac, image, name, experiment, variant, prompt, strength):\n        image.save(out / name)\n        score = sketch_edge_recall(control, image, int(s.get(\"edge_tolerance_pixels\", 5)))\n        rows.append({\"sketch\": e[\"sketch\"], \"experiment\": experiment, \"prompt_variant\": variant,\n                     \"result\": f\"outputs/{name}\", \"control_strength\": strength,\n                     \"edge_recall_proxy\": round(score, 4), \"prompt\": prompt,\n                     \"negative_prompt\": s.get(\"negative_prompt\"), \"seed\": seed,\n                     \"steps\": int(s[\"num_inference_steps\"]), \"guidance_scale\": float(s[\"guidance_scale\"]),\n                     \"threshold\": int(s.get(\"threshold\", 205)), \"ink_fraction\": round(frac, 5),\n                     \"source_sha256\": sha256(sketch), \"model_base\": s[\"model_base\"],\n                     \"controlnet_model\": s[\"controlnet_model\"], \"device\": device,\n                     \"created_utc\": datetime.now(timezone.utc).isoformat()})\n        export_rows(rows, out)   # I rewrite the records after every image, so a crash loses nothing\n        print(\"Saved\", name, \"edge proxy\", f\"{score:.3f}\")\n\n    for e in s[\"experiments\"]:\n        sketch, control, frac = controls[e[\"sketch\"]]\n        stem = Path(e[\"sketch\"]).stem\n\n        # Experiment 1: same prompt, three control strengths.\n        cells, main_at_variant_strength = [], None\n        for strength in [float(v) for v in e[\"control_strengths\"]]:\n            image = generate(e, control, e[\"prompt\"], strength)\n            record(e, sketch, control, frac, image, f\"{stem}_control_{strength:.2f}.png\",\n                   \"control_strength\", \"main\", e[\"prompt\"], strength)\n            cells.append((f\"Scale {strength:.2f}\" if strength else \"Baseline (no control)\", image))\n            if abs(strength - variant_strength) < 1e-9:\n                main_at_variant_strength = image\n        make_comparison(sketch, control, cells, out / f\"{stem}_comparison.png\")\n\n        # Experiment 2: same sketch, seed and strength, different prompts.\n        variants = e.get(\"prompt_variants\", [])\n        if variants:\n            if main_at_variant_strength is None:\n                main_at_variant_strength = generate(e, control, e[\"prompt\"], variant_strength)\n                record(e, sketch, control, frac, main_at_variant_strength,\n                       f\"{stem}_prompt_main.png\", \"prompt\", \"main\", e[\"prompt\"], variant_strength)\n            prompt_cells = [(f\"Main prompt ({variant_strength:.2f})\", main_at_variant_strength)]\n            for v in variants:\n                image = generate(e, control, v[\"prompt\"], variant_strength)\n                record(e, sketch, control, frac, image, f\"{stem}_prompt_{v['name']}.png\",\n                       \"prompt\", v[\"name\"], v[\"prompt\"], variant_strength)\n                prompt_cells.append((f\"Prompt: {v['name']}\", image))\n            make_comparison(sketch, control, prompt_cells, out / f\"{stem}_prompts.png\")\n    return rows\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=\"My sketch-to-image experiments\")\n    ap.add_argument(\"--config\", type=Path, default=Path(\"experiments.json\"))\n    mode = ap.add_mutually_exclusive_group(required=True)\n    mode.add_argument(\"--dry-run\", action=\"store_true\", help=\"Check my sketches and save control maps; no download\")\n    mode.add_argument(\"--generate\", action=\"store_true\", help=\"Run real inference with the pretrained models\")\n    ap.add_argument(\"--allow-cpu\", action=\"store_true\", help=\"CPU generation works but is very slow\")\n    args = ap.parse_args()\n    root = Path(__file__).resolve().parents[1]\n    config = args.config if args.config.is_absolute() else root / args.config\n    s = load_settings(config)\n    out = root / \"outputs\"\n    out.mkdir(parents=True, exist_ok=True)\n    controls = prepare_inputs(s, root, out)\n    if args.dry_run:\n        print(\"Dry run only: no models loaded and no images generated.\")\n        return\n    pipe, device = load_pipeline(s, args.allow_cpu)\n    import torch\n    (out / \"environment.json\").write_text(json.dumps({\n        \"packages\": packages(), \"device\": device,\n        \"cuda_device\": torch.cuda.get_device_name(0) if device == \"cuda\" else None,\n        \"configuration\": s}, indent=2), encoding=\"utf-8\")\n    rows = run_generation(s, root, out, controls, pipe, device,\n                          lambda seed: torch.Generator(device=\"cpu\").manual_seed(seed))\n    print(f\"Finished {len(rows)} images. Next: look at them and fill in human_review_template.csv.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "experiments.json": "{\n  \"model_base\": \"stable-diffusion-v1-5/stable-diffusion-v1-5\",\n  \"controlnet_model\": \"lllyasviel/control_v11p_sd15_scribble\",\n  \"negative_prompt\": \"poor geometry, blurry, distorted, unreadable text, watermark\",\n  \"size\": 512,\n  \"num_inference_steps\": 25,\n  \"guidance_scale\": 7.0,\n  \"seed\": 2026,\n  \"threshold\": 205,\n  \"edge_tolerance_pixels\": 5,\n  \"prompt_variant_strength\": 0.65,\n  \"experiments\": [\n    {\n      \"sketch\": \"01_car.png\",\n      \"label\": \"car\",\n      \"prompt\": \"a modern car, realistic photograph, glossy red paint, chrome details, parked on a city street, daylight, high detail\",\n      \"control_strengths\": [\n        0.0,\n        0.65,\n        1.1\n      ],\n      \"prompt_variants\": [\n        {\n          \"name\": \"vintage\",\n          \"prompt\": \"a vintage 1960s car, black and white film photograph, quiet country road, soft overcast light\"\n        },\n        {\n          \"name\": \"futuristic\",\n          \"prompt\": \"a futuristic electric concept car, neon-lit city street at night, wet road reflections, glossy 3D render\"\n        }\n      ]\n    },\n    {\n      \"sketch\": \"02_house.png\",\n      \"label\": \"house\",\n      \"prompt\": \"a two-storey family house, realistic architectural photograph, brick walls, tiled roof, front garden, daylight, high detail\",\n      \"control_strengths\": [\n        0.0,\n        0.65,\n        1.1\n      ],\n      \"prompt_variants\": [\n        {\n          \"name\": \"snow_cabin\",\n          \"prompt\": \"a wooden cabin in a snowy pine forest at dusk, warm light in the windows, realistic photograph\"\n        },\n        {\n          \"name\": \"watercolour\",\n          \"prompt\": \"a watercolour illustration of a cottage with a flower garden, soft pastel colours, paper texture\"\n        }\n      ]\n    },\n    {\n      \"sketch\": \"03_sword.png\",\n      \"label\": \"sword\",\n      \"prompt\": \"a medieval steel longsword, realistic product photograph, polished blade, leather-wrapped grip, plain dark background, studio lighting, high detail\",\n      \"control_strengths\": [\n        0.0,\n        0.65,\n        1.1\n      ],\n      \"prompt_variants\": [\n        {\n          \"name\": \"crystal\",\n          \"prompt\": \"a fantasy sword made of glowing blue crystal, dark cave background, dramatic rim lighting\"\n        },\n        {\n          \"name\": \"rusted\",\n          \"prompt\": \"an ancient rusted iron sword half-buried in desert sand, archaeological find, natural daylight\"\n        }\n      ]\n    }\n  ]\n}\n",
 "requirements.txt": "# I install a CUDA build of PyTorch first if my machine needs one (Colab already has it).\n# The exact versions from my real run are recorded in outputs/environment.json.\ntorch>=2.3,<3\ndiffusers>=0.35.1,<0.40\ntransformers>=4.45,<5\naccelerate>=1,<2\nhuggingface-hub>=0.25,<2\nsafetensors>=0.4,<1\nPillow>=10,<13\nnumpy>=1.26,<3\npytest>=8,<10\njupyterlab>=4,<5\n",
 "README.md": "# Bringing Sketches to Life with Generative AI\n\n**Author:** Nosa Peter Inwe\n**Course:** Computer Vision and Machine Learning II, Université Paris-Est Créteil (UPEC)\n**Professor:** Arezoo Ghodsifard\n**Submission deadline:** 20 November 2026\n\nI drew three sketches by hand (a car, a house and a sword) and built a Python pipeline that turns each one into a finished image with Stable Diffusion 1.5 and a Scribble ControlNet. My question is simple: how much of my drawing's structure survives when the model invents the materials, lighting and background? I test it two ways, with a fixed seed throughout so the starting noise never changes between runs.\n\n## How the sketch steers the image\n\nStable Diffusion 1.5 is a latent diffusion model. It does not paint pixels directly: a VAE encoder compresses a 512×512 image into a 64×64×4 latent, and a UNet learns to remove noise from that latent step by step. At generation time I start from pure Gaussian noise (fixed by the seed), and the UNet, guided by my text prompt through the CLIP text encoder, denoises it over 25 scheduler steps. The VAE decoder then turns the clean latent back into pixels.\n\nOn its own, the text prompt only says *what* to draw, not *where*. That is the job of ControlNet. It is a trainable copy of the UNet's encoder that reads a spatial condition image (for the scribble model, white strokes on black) and adds its features back into the UNet at every denoising step through zero-initialised convolutions. The `controlnet_conditioning_scale` multiplies those added features:\n\n- **0.00:** the ControlNet contributes nothing. The image comes from the prompt alone, so it is my baseline.\n- **0.65:** moderate guidance. The layout should follow my lines but the model keeps some freedom.\n- **1.10:** strong guidance. The lines should dominate, sometimes at the cost of realism.\n\nBefore any of that, my preprocessing (`src/sketch_pipeline.py`) turns the photo of my drawing into the control map. It applies the phone's rotation flag, pads the image to a square without stretching it, converts to greyscale, and marks every pixel darker than a threshold (205/255 by default) as a white stroke on black.\n\n![My pipeline](assets/pipeline.png)\n\n## My experiments\n\n| Experiment | What changes | What stays fixed | Images |\n|---|---|---|---|\n| 1. Control strength | ControlNet scale 0.00 / 0.65 / 1.10 | prompt, seed, steps, guidance, resolution | 3 per sketch |\n| 2. Prompt variation | two alternative prompts | sketch, seed, scale (0.65), steps, guidance | 2 per sketch (+ the main prompt at 0.65 from experiment 1) |\n\nThat's 15 generated images in total. Changing one thing at a time is the point: if I changed the prompt and the strength together, I could not say which one caused a difference.\n\n| Sketch | Main prompt | Variant 1 | Variant 2 |\n|---|---|---|---|\n| Car | modern red car on a city street, daylight | vintage 1960s car, black-and-white film photo | futuristic electric concept car, neon city at night |\n| House | two-storey brick house with a front garden | wooden cabin in a snowy forest at dusk | watercolour cottage with a flower garden |\n| Sword | medieval steel longsword, studio product shot | glowing blue crystal fantasy sword | rusted iron sword half-buried in sand |\n\nThe exact prompt text lives in `experiments.json`.\n\n## Settings I use\n\n| Setting | Value |\n|---|---|\n| Base model | [`stable-diffusion-v1-5/stable-diffusion-v1-5`](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) |\n| Control model | [`lllyasviel/control_v11p_sd15_scribble`](https://huggingface.co/lllyasviel/control_v11p_sd15_scribble) (ControlNet v1.1 Scribble) |\n| Scheduler | UniPC multistep, 25 steps |\n| Classifier-free guidance | 7.0 |\n| Resolution | 512 × 512 |\n| Seed | 2026 (fresh CPU generator for every image) |\n| Negative prompt | poor geometry, blurry, distorted, unreadable text, watermark |\n| Sketch threshold | 205 / 255 |\n| Precision | float16 on GPU |\n\n## Running it\n\n### On Google Colab (what I used)\n\n1. Upload `notebooks/01_sketch_to_image.ipynb` to [Colab](https://colab.research.google.com) and choose **Runtime → Change runtime type → T4 GPU**.\n2. Run the cells in order. The project code is built into the notebook, so the only upload is my three sketches (file names containing `car`, `house` and `sword`).\n3. The dry-run cell shows each sketch beside its control map. I check them before generating; if a shadow shows up as a white blob I lower the threshold, and if lines go missing I raise it.\n4. The generation cell downloads the models from Hugging Face on the first run, then makes the 15 images.\n5. The last cell downloads `sketch_results.zip` with everything in `outputs/`.\n\n### Locally (needs a CUDA GPU with enough memory; my laptop's 3 GB card is not enough)\n\n```bash\npython -m venv .venv\n.venv\\Scripts\\activate            # macOS/Linux: source .venv/bin/activate\npython -m pip install -r requirements.txt\npython -m pytest -q\npython -m src.run_experiments --dry-run\npython -m src.run_experiments --generate\n```\n\nMy drawings go in `data/sketches/` as `01_car.png`, `02_house.png` and `03_sword.png`.\n\n## What a run produces (`outputs/`)\n\n- `*_control.png`: the processed control map for each sketch.\n- `*_control_0.00.png`, `*_control_0.65.png`, `*_control_1.10.png`: experiment 1.\n- `*_prompt_<name>.png`: experiment 2.\n- `*_comparison.png` and `*_prompts.png`: side-by-side sheets (my sketch, control map, results).\n- `run_manifest.json` and `run_summary.csv`: every prompt, seed, scale, model ID and the SHA-256 of each input sketch.\n- `environment.json`: exact library versions and the GPU name.\n- `human_review_template.csv`: where I score each image 1–5 for structure, image quality and prompt fit, with one concrete strength and failure.\n\n## How I judge the results\n\nMostly by eye, against my own drawing. A 5 for structure means the main silhouette and the spatial relations between parts survive; a 3 means some features match but important geometry moved; a 1 means the shape is gone. A beautiful image that ignores my sketch does not get a high structure score.\n\nI also log an *edge-proximity proxy*: the share of my stroke pixels that sit within 5 pixels of an edge in the generated image. It helps me compare strengths on the same sketch, but it is crude. Busy textures like brick or foliage create edges everywhere and inflate it, so I never treat it as a similarity score.\n\n## Limits I am aware of\n\n- A fixed seed makes runs repeatable on the same setup, but different GPUs or library versions can still give slightly different images.\n- Shadows in a phone photo can turn into fake strokes; I check the control maps before generating.\n- Stable Diffusion 1.5 has known weaknesses (hands, text, fine detail) and inherits biases from its training data.\n- The scribble model was trained on rough line drawings, so very detailed shading in a sketch does not help it.\n- My tests (`tests/`) check preprocessing, configuration and bookkeeping with a fake pipeline. They do not test image quality; only the real run does that.\n\n## Optional local interface\n\n`app.py` is a small Streamlit page where I can upload a sketch, type a prompt and pick a strength. It needs a CUDA GPU:\n\n```bash\npip install -r requirements-ui.txt\nstreamlit run app.py\n```\n\n## References\n\n- Rombach, R., Blattmann, A., Lorenz, D., Esser, P. & Ommer, B. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. CVPR. https://arxiv.org/abs/2112.10752\n- Zhang, L., Rao, A. & Agrawala, M. (2023). *Adding Conditional Control to Text-to-Image Diffusion Models*. ICCV. https://arxiv.org/abs/2302.05543\n- Ho, J., Jain, A. & Abbeel, P. (2020). *Denoising Diffusion Probabilistic Models*. NeurIPS. https://arxiv.org/abs/2006.11239\n- Zhao, W. et al. (2023). *UniPC: A Unified Predictor-Corrector Framework for Fast Sampling of Diffusion Models*. https://arxiv.org/abs/2302.04867\n- Stable Diffusion v1.5 model card and CreativeML OpenRAIL-M licence: https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5\n- ControlNet v1.1 Scribble model card: https://huggingface.co/lllyasviel/control_v11p_sd15_scribble\n- Hugging Face Diffusers, ControlNet pipeline: https://huggingface.co/docs/diffusers/api/pipelines/controlnet\n- Hugging Face Diffusers, reproducibility and seeds: https://huggingface.co/docs/diffusers/using-diffusers/reusing_seeds\n- Libraries: PyTorch, Diffusers, Transformers, Accelerate, Pillow, NumPy, pytest, Streamlit. Exact versions for my run are in `outputs/environment.json`.\n- Sketches: all three are my own drawings.\n\n## AI assistance\n\nI used an AI coding assistant (Claude, by Anthropic) to help write and test parts of the pipeline code, the notebook and this README. I chose the subjects, drew the sketches, ran the experiments, judged the results and wrote the analysis myself, and I can explain every part of the code.\n"
}

if IN_COLAB:
    ROOT = Path("/content/sketch-to-image-controlnet-lab")
    for rel, text in PROJECT_FILES.items():
        dest = ROOT / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(text, encoding="utf-8")
    (ROOT / "data" / "sketches").mkdir(parents=True, exist_ok=True)
    (ROOT / "outputs").mkdir(parents=True, exist_ok=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Code files ready:", ", ".join(PROJECT_FILES))

In [ ]:
# 3. I install the diffusion libraries (same version ranges as requirements.txt; Colab already has PyTorch)
pkgs = ["diffusers>=0.35.1,<0.40", "transformers>=4.45,<5", "accelerate>=1,<2",
        "huggingface-hub>=0.25,<2", "safetensors>=0.4,<1"]
run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
import torch, diffusers, transformers
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| diffusers", diffusers.__version__, "| transformers", transformers.__version__)

In [ ]:
# 4. I upload my three sketches. Each file name contains 'car', 'house' or 'sword'
# (e.g. car.jpg), which is how I match it to its prompt. I save them as PNG
# under the names experiments.json expects.
import json
from PIL import Image, ImageOps

cfg = json.loads((ROOT / "experiments.json").read_text(encoding="utf-8"))
sketch_dir = ROOT / "data" / "sketches"
sketch_dir.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        match = [e for e in cfg["experiments"] if e["label"] in name.lower()]
        if not match:
            print(f"Skipped {name}: its name does not contain car, house or sword")
            continue
        with Image.open(name) as im:
            ImageOps.exif_transpose(im).convert("RGB").save(sketch_dir / match[0]["sketch"])
        print(f"{name} -> data/sketches/{match[0]['sketch']}")

missing = [e["sketch"] for e in cfg["experiments"] if not (sketch_dir / e["sketch"]).exists()]
print("Missing:", missing) if missing else print("All three sketches are in place.")

## 5. Dry run: I check the control maps before generating

I pad each sketch to a square, convert it to greyscale and threshold it: pixels darker than `THRESHOLD` (default 205) become white strokes on black. This map is what ControlNet actually sees, so I check it by eye. Shadows from a phone photo show up as white blobs, and faint pencil can vanish.

- Blobs or grey patches: I lower the threshold (e.g. 170) or retake the photo in better light.
- Lines missing: I raise it (e.g. 225) or go over the lines in darker pen.

In [ ]:
# I change the threshold here if a map looks wrong, then re-run this cell
THRESHOLD = 205

cfg["threshold"] = THRESHOLD
(ROOT / "experiments.json").write_text(json.dumps(cfg, indent=2) + "\n", encoding="utf-8")
run([sys.executable, "-m", "src.run_experiments", "--dry-run"], cwd=ROOT)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, e in enumerate(cfg["experiments"]):
    stem = Path(e["sketch"]).stem
    axes[0, i].imshow(Image.open(sketch_dir / e["sketch"])); axes[0, i].set_title(f"{e['label']}: original")
    axes[1, i].imshow(Image.open(ROOT / "outputs" / f"{stem}_control.png")); axes[1, i].set_title("control map")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 6. I generate the 15 images

The first run downloads Stable Diffusion 1.5 and the Scribble ControlNet from Hugging Face (a few GB). After that each image takes seconds on a T4.

Two experiments, both with seed 2026 so the starting noise never changes:
- **Control strength:** the main prompt at scales 0.00 (prompt only, my baseline), 0.65 and 1.10.
- **Prompt variation:** two different prompts at 0.65, compared with the main prompt at 0.65.

If I want to adjust a prompt, I do it in the cell below *before* generating and keep it fixed across that sketch's runs, so each comparison changes one thing at a time.

In [ ]:
# Prompt overrides go here if I need them, e.g.
# cfg["experiments"][0]["prompt"] = "a vintage blue convertible car, realistic photograph, ..."
# cfg["experiments"][0]["prompt_variants"][0]["prompt"] = "..."
(ROOT / "experiments.json").write_text(json.dumps(cfg, indent=2) + "\n", encoding="utf-8")
for e in cfg["experiments"]:
    print(f"{e['label']:>6} main: {e['prompt']}")
    for v in e.get("prompt_variants", []):
        print(f"{'':>6} {v['name']}: {v['prompt']}")

run([sys.executable, "-m", "src.run_experiments", "--generate"], cwd=ROOT)

## 7. My results

For each sketch I get two sheets:
- `*_comparison.png`: my sketch, its control map, then scales 0.00, 0.65 and 1.10 with the main prompt.
- `*_prompts.png`: my sketch, its control map, then the main prompt and the two variants, all at 0.65.

The `edge_recall_proxy` is only a rough helper: busy textures can push it up without the shape being right. My real judgement goes into `outputs/human_review_template.csv`, where I score each image 1–5 for structure, image quality and prompt fit, with one concrete strength and failure.

In [ ]:
from IPython.display import display
import csv
for e in cfg["experiments"]:
    for kind in ("comparison", "prompts"):
        sheet = ROOT / "outputs" / f"{Path(e['sketch']).stem}_{kind}.png"
        if sheet.exists():
            print(f"{e['label']}: {kind}"); display(Image.open(sheet))

summary = ROOT / "outputs" / "run_summary.csv"
if summary.exists():
    for row in csv.DictReader(summary.open(encoding="utf-8")):
        print(f"{row['sketch']:<13} {row['experiment']:<16} {row['prompt_variant']:<12} "
              f"scale={float(row['control_strength']):.2f}  edge_proxy={row['edge_recall_proxy']}")
else:
    print("No generated results yet.")

In [ ]:
# 8. I download my sketches and all outputs (15 images, comparison sheets, manifest, environment, review sheet)
import zipfile
bundle = Path("/content/sketch_results.zip") if IN_COLAB else ROOT / "sketch_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT / "experiments.json", "experiments.json")
    for folder in ("data/sketches", "outputs"):
        for f in sorted((ROOT / folder).glob("*")):
            if f.is_file() and f.name != ".gitkeep":
                z.write(f, f"{folder}/{f.name}")
print("Created", bundle, "with", len(zipfile.ZipFile(bundle).namelist()), "files")
if IN_COLAB:
    from google.colab import files
    files.download(bundle)